# Business Performance & Profitability Analysis

**Portfolio project · Python · Exploratory data analysis**

This notebook examines sales, revenue, cost and profit data to demonstrate data-quality checks, feature engineering, KPI calculation and comparisons across time and product categories.

## Questions explored
- How do revenue and profit vary by year?
- Which product categories contribute most to revenue and profit?
- What additional features can support segmentation and follow-up analysis?

## Reproducibility
Keep `Sales Dataset (1).xlsx` in the same directory as this notebook. Install the packages in `requirements.txt`, then run cells from top to bottom. The original notebook's personal computer path has been replaced with a portable project-relative path.

## Interpretation notes
Results describe this dataset only. Confirm the dataset source and licence before redistribution, and interpret profit margin, age-band definitions and duplicate removal in light of documented data definitions.


## 1. Data loading and quality checks

Inspect the dataset's dimensions, fields, missing values and duplicate rows before calculating metrics. Any data-cleaning decision should be justified and recorded.


## 2. Feature engineering

Derived fields include profit margin, calendar month and quarter, customer age band and a high-value-sale indicator. Validate definitions before using these features for business decisions.


## 3. Key performance indicators

Summary metrics are calculated from the cleaned records. Distinguish the mean of row-level margins from the ratio of total profit to total revenue; these answer different questions.


## 4. Time and category analysis

The following sections compare yearly performance and product-category totals. Use the resulting plots as descriptive evidence rather than as proof of causal drivers.


In [ ]:
# Core libraries for reproducible exploratory analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the dataset from the repository folder
from pathlib import Path

DATA_PATH = Path("Sales Dataset (1).xlsx")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the workbook in the same folder as this notebook."
    )

df = pd.read_excel(DATA_PATH)
df.head()

In [ ]:
#DATAFRAME STRUCTURE AND DIMENSION
df.info()
df.shape
df.head()



In [ ]:
#COLUMS NAMES
df.columns

In [ ]:
# DATA CLEANSING AND VALIDATION

# Check missing values
df.isnull().sum()



In [ ]:
# Check duplicates(There are 1,178 duplicate rows in the dataset)
df.duplicated().sum()

In [ ]:
#REMOVING DUPLICATE ROWS
df= df.drop_duplicates()

In [ ]:
#CHECKING FOR DUPLICATE ON THE CLEAN DATA
df.duplicated().sum()


In [ ]:
#CHECKING DATA SIZE AFTER REMOVING DUPLICATES

df.shape


In [ ]:
#EXPORT TO EXCEL


In [ ]:
# Summary statistics for numerical columns
df.describe()


In [ ]:
# Optional export: enable only after confirming the dataset licence permits redistribution.
# df.to_excel("salesdata_cleaned.xlsx", index=False)

In [ ]:
df = df.copy()


In [ ]:
# Confirm that the source workbook is available in the project directory
from pathlib import Path
print(f"Dataset found: {Path('Sales Dataset (1).xlsx').exists()}")

In [ ]:
#CREATING NEW COLUMN(profit Margin)
df[ "Profit_Margin"] = df["Profit"] / df["Revenue"]




In [ ]:
# Extract month number for proper sorting

df["Month_Number"] = df["Date"].dt.month
df.head()

In [ ]:
# Extract quarter
df["Quarter"] = df["Date"].dt.to_period("Q").astype(str)
df.head()

In [ ]:
# Customer age bands with labels aligned to the numerical intervals
df["Age_Band"] = pd.cut(
    df["Customer_Age"],
    bins=[0, 17, 24, 34, 44, 54, 120],
    labels=["Under 18", "18–24", "25–34", "35–44", "45–54", "55+"],
    include_lowest=True
)

In [ ]:
df[["Customer_Age", "Age_Band"]].head(10)

In [ ]:
#Revenue Contribution(TOP 25% OF SALES BY REVENUE)
# Flag high-value transactions
df["High_Value_Sale"] = df["Revenue"] > df["Revenue"].quantile(0.75)
df.head()

In [ ]:
df.head()

In [ ]:
#Validate New Columns
df.columns


In [ ]:
# Handle undefined profit margins where revenue is zero
# These rows are set to 0 for this exploratory summary; document this choice when interpreting results.
df["Profit_Margin"] = (
    df["Profit_Margin"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [ ]:
#Key Business KPIs (Executive Summary)

total_revenue = df["Revenue"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order_Quantity"].sum()
avg_profit_margin = df["Profit_Margin"].mean()

total_revenue, total_profit, total_orders, avg_profit_margin


In [ ]:
#Revenue & Profit Trend Over Time
yearly_trend = df.groupby("Year")[["Revenue", "Profit"]].sum().reset_index()
yearly_trend


In [ ]:
import matplotlib as plt
import matplotlib.pyplot as plt


plt.figure()
plt.plot(yearly_trend["Year"], yearly_trend["Revenue"], label="Revenue")
plt.plot(yearly_trend["Year"], yearly_trend["Profit"], label="Profit")
plt.xlabel("Year")
plt.ylabel("Amount")
plt.title("Revenue and Profit Trend Over Time")
plt.legend()
plt.show()


In [ ]:
#Sales by Product Category
category_sales = (
    df.groupby("Product_Category")[["Revenue", "Profit"]]
    .sum()
    .sort_values("Revenue", ascending=False)
)

category_sales


In [ ]:
category_sales.plot(kind="bar")
plt.title("Revenue and Profit by Product Category")
plt.ylabel("Amount")
plt.xlabel("Product Category")
plt.show()


In [ ]:
# Top 10 Products by Profit
top_products = (
    df.groupby("Product")["Profit"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products



In [ ]:
#TOP 10 PRODUCTS BY PROFIT
top_products.plot(kind="barh")
plt.title("Top 10 Products by Profit")
plt.xlabel("Profit")
plt.show()


In [ ]:
#Customer Demographics Analysis--Revenue by Age group
age_revenue = df.groupby("Age group")["Revenue"].sum().sort_values(ascending=False)
age_revenue




In [ ]:
age_revenue.plot(kind="bar")
plt.title("Revenue by Age Group")
plt.ylabel("Revenue")
plt.show()


In [ ]:
#🔹 Revenue by Gender
gender_revenue = df.groupby("Customer_Gender")["Revenue"].sum()
gender_revenue


In [ ]:
gender_revenue.plot(kind="pie", autopct="%1.1f%%")
plt.title("Revenue Share by Gender")
plt.ylabel("")
plt.show()


In [ ]:
#Seasonality Analysis
season_sales = df.groupby("Season")["Revenue"].sum().sort_values(ascending=False)
season_sales


In [ ]:
season_sales.plot(kind="bar")
plt.title("Revenue by Season")
plt.ylabel("Revenue")
plt.show()


In [ ]:
#Monthly Trend
monthly_trend = (
    df.groupby(["Month_Number", "Month"])["Revenue"]
    .sum()
    .reset_index()
    .sort_values("Month_Number")
)

monthly_trend


In [ ]:
plt.figure()
plt.plot(monthly_trend["Month"], monthly_trend["Revenue"])
plt.xticks(rotation=45)
plt.title("Monthly Revenue Trend")
plt.ylabel("Revenue")
plt.show()


In [ ]:
df.head()


In [ ]:
# Final export for Power BI
df.to_excel("sales_data.xlsx", index=False)

In [ ]:
C:\Users\HP\sales_data.xlsx